<a href="https://colab.research.google.com/github/YassGan/3DGaussianSplatting-INRIA-Method-Colab/blob/feat%2Fworking_with_drive_ds/working_with_drive_ds/working_with_drive_ds/working_with_drive_ds/Copy_of_3DGaussianSplatting_INRIA_Method_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Making sure that we are using a GPU

In [ ]:
!nvidia-smi

Wed Mar  5 23:42:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Inputting the zip file name (should be located in the drive) of the images which we want to create a 3d scene from

In [ ]:
images_zip_file_name="RAW_Images"

# Installing pycolmap and colmap before any other installation or modificattion

In [ ]:
# Install COLMAP in Colab
print("Installing COLMAP...")
!apt-get install -y colmap
!pip install pycolmap

Installing COLMAP...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libamd2 libcamd2 libccolamd2 libceres2 libcholmod3 libcolamd2 libcxsparse3 libevdev2
  libfreeimage3 libgflags2.2 libgoogle-glog0v5 libgudev-1.0-0 libinput-bin libinput10 libjxr0
  libmd4c0 libmetis5 libmtdev1 libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5
  libqt5widgets5 libraw20 libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common libwacom9
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1 libxcb-xinerama0
  libxcb-xinput0 libxcb-xkb1 libxkbcommon-x11-0 qt5-gtk-platformtheme qttranslations5-l10n
Suggested packages:
  qt5-image-formats-plugins qtwayland5
The following NEW packages will be installed:
  colmap libamd2 libcamd2 libccolamd2 libceres2 libcholmod3 libcolamd2 libcxsparse3 libevdev2
  libfreeimage3 libgflags2.2 libgoogle-glog0v5 libgudev-1.0-0 libinput-bi

# Python downgrading


In [ ]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version  # Should say Python 3.7.x

--2025-03-05 23:42:24--  https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 90665082 (86M) [application/x-sh]
Saving to: ‘mini.sh’

mini.sh             100%[===================>]  86.46M   194MB/s    in 0.4s    

2025-03-05 23:42:25 (194 MB/s) - ‘mini.sh’ saved [90665082/90665082]

PREFIX=/usr/local
Unpacking payload ...
                                                                                 
Installing base environment...





Preparing transaction: - \ | done
Executing transaction: - \ | / - \ | / - \ | / - \ | / - \ | done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the 

# CUDA 11.8

In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8

--2025-03-05 23:42:54--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.46.228.167, 23.46.228.176, 23.46.228.173, ...
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.46.228.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G   136MB/s    in 45s     

2025-03-05 23:43:39 (91.2 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


#Pytorch with cuda

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 615.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 60.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 56.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 75.5 MB/s eta 0:00:00


# Verification of the versions

In [ ]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)        # Should be 11.3 (from PyTorch)
print(torch.cuda.get_device_name(0))  # Should show GPU

True
12.4
Tesla T4


In [ ]:
##making sure that we are always using the GPU and not a CPU
!nvidia-smi

Wed Mar  5 23:49:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Cloning the 3D Gaussian Splatting algorithm and the submodules of the algorithm

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 28.16 MiB/s, done.
Resolving deltas: 100% (349/349), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (174/174), done.        
remote: Total 3293 (delta 171), reused 280 (delta 148), pack-reused 2971 (from 1)        
Receiving objects:

# Mounting the drive

In [ ]:

images_folder_name="Images"

import os
import zipfile
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
# Paths




Mounted at /content/drive


# Choosing the file

In [ ]:
!unzip "/content/drive/MyDrive/{images_zip_file_name}.zip" -d "/content/{images_zip_file_name}"


Archive:  /content/drive/MyDrive/RAW_Images.zip
replace /content/RAW_Images/RAW_Images/DSC05572.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05573.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05574.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05575.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05576.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05577.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05578.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05579.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05580.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_Images/RAW_Images/DSC05581.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/RAW_I

COLMAP Work with RAW images zip file

In [ ]:
from pathlib import Path
import os
import subprocess
import pycolmap



print(images_zip_file_name)


# Define paths
output_path = Path("COLMAP_Output")  # Output directory for final reconstruction
image_dir = f"/content/{images_zip_file_name}/{images_zip_file_name}"     # Input images directory
initial_recon_path = output_path / "initial"  # Temporary directory for initial reconstruction



# Create directories if they don’t exist
output_path.mkdir(exist_ok=True)
initial_recon_path.mkdir(exist_ok=True)
database_path = output_path / "database.db"
undistorted_output = output_path / "undistorted"

# Step 1: Initial SfM pipeline to generate a reconstruction
print("Running initial feature extraction...")
pycolmap.extract_features(database_path, image_dir)

print("Running initial feature matching...")
pycolmap.match_exhaustive(database_path)

print("Running initial incremental mapping...")
maps = pycolmap.incremental_mapping(database_path, image_dir, initial_recon_path)
maps[0].write(initial_recon_path)

# Step 2: Undistort images using the initial reconstruction
print("Running image undistortion...")
subprocess.run([
    "colmap", "image_undistorter",
    "--image_path", str(image_dir),
    "--input_path", str(initial_recon_path),  # Use initial reconstruction
    "--output_path", str(undistorted_output),
    "--output_type", "COLMAP",
    "--max_image_size", "2000"  # Optional: adjust based on image resolution
], check=True)

# Update paths to use undistorted images
undistorted_image_dir = undistorted_output / "images"

# Step 3: Re-run SfM pipeline on undistorted images
print("Extracting features from undistorted images...")
pycolmap.extract_features(database_path, undistorted_image_dir)

print("Matching features from undistorted images...")
pycolmap.match_exhaustive(database_path)

print("Running incremental mapping on undistorted images...")
maps = pycolmap.incremental_mapping(database_path, undistorted_image_dir, output_path)
maps[0].write(output_path)

# Step 4: Verify camera models (fixed path to match output_path)
print("Verifying camera models in the output...")
reconstruction = pycolmap.Reconstruction("content/COLMAP_Output/undistorted/sparse")  # Corrected from "outputIM/undistorted/sparse"
for cam_id, cam in reconstruction.cameras.items():
    print(cam)

RAW_Images
Running initial feature extraction...
Running initial feature matching...
Running initial incremental mapping...
Running image undistortion...
Extracting features from undistorted images...
Matching features from undistorted images...
Running incremental mapping on undistorted images...
Verifying camera models in the output...
Camera(camera_id=31, model=SIMPLE_RADIAL, width=1264, height=832, params=[1038.99, 632, 416, -0.00171648] (f, cx, cy, k))
Camera(camera_id=30, model=SIMPLE_RADIAL, width=1264, height=832, params=[1036.56, 632, 416, 0.00194525] (f, cx, cy, k))
Camera(camera_id=13, model=SIMPLE_RADIAL, width=1264, height=832, params=[1056.1, 632, 416, 0.00842877] (f, cx, cy, k))
Camera(camera_id=12, model=SIMPLE_RADIAL, width=1264, height=832, params=[1053.32, 632, 416, 0.00160235] (f, cx, cy, k))
Camera(camera_id=11, model=SIMPLE_RADIAL, width=1264, height=832, params=[1052, 632, 416, 0.00535985] (f, cx, cy, k))
Camera(camera_id=10, model=SIMPLE_RADIAL, width=1264, heig

In [ ]:
print(output_path)

COLMAP_Output


# Arranging Input folder for the 3D Gaussian INRIA

In [ ]:
import os
import shutil
from pathlib import Path

# Define paths (adjust these to your actual paths)



input_images_folder = f"/content/{output_path}/undistorted/images/" # Folder with your input images
colmap_output_folder =  f"/content/{output_path}/undistorted/sparse/"   # Folder containing COLMAP's output (e.g., "sparse" files)
new_parent_folder = "3D_Gaussian_Splatting_input_folder2"  # New folder to create

# Create the new parent folder
new_parent = Path(new_parent_folder)
new_parent.mkdir(parents=True, exist_ok=True)

# 1. Copy input images to "images" subfolder
images_subfolder = new_parent / "images"
images_subfolder.mkdir(exist_ok=True)

# Copy all images from input_images_folder to images_subfolder
for img in Path(input_images_folder).glob("*"):
    if img.is_file() and img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        shutil.copy(img, images_subfolder / img.name)

# 2. Create "sparse" subfolder and copy COLMAP output files
sparse_subfolder = new_parent / "sparse/0/"
# Create the parent directory 'sparse' first
sparse_subfolder.parent.mkdir(parents=True, exist_ok=True)
# Now create the '0' subfolder
sparse_subfolder.mkdir(exist_ok=True)


# Copy COLMAP reconstruction files (cameras.bin, images.bin, points3D.bin)
required_colmap_files = ["cameras.bin", "images.bin", "points3D.bin"]
for file in required_colmap_files:
    src = Path(colmap_output_folder) / file
    if src.exists():
        shutil.copy(src, sparse_subfolder / file)
    else:
        print(f"Warning: {file} not found in COLMAP output folder!")

print(f"Dataset folder created at: {new_parent_folder}")

Dataset folder created at: 3D_Gaussian_Splatting_input_folder2


# Training the model

In [ ]:
!python /content/gaussian-splatting/train.py \
  -s /content/3D_Gaussian_Splatting_input_folder2// \
  -m /content/output \
  --iterations 30000 \
  --test_iterations 1000 2000 3000 4000 5000 \
  --save_iterations 1000 2000 3000 4000 5000


n
Optimizing /content/output
Output folder: /content/output [06/03 00:16:09]
Tensorboard not available: not logging progress [06/03 00:16:09]
Reading camera 31/31 [06/03 00:16:10]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [06/03 00:16:10]
Loading Training Cameras [06/03 00:16:10]
Loading Test Cameras [06/03 00:16:17]
Number of points at initialisation :  5419 [06/03 00:16:17]
Training progress:   0% 100/30000 [00:04<20:28, 24.35it/s, Loss=0.1780452]Traceback (most recent call last):
  File "/content/gaussian-splatting/train.py", line 216, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/content/gaussian-splatting/train.py", line 90, in training
    loss.backward()
  File "/usr/local/lib/python3.7/site-packages/torch/_tensor.py", line 396, in backward
    torch.autograd.backward(self, gradient, re